<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 4.2 — Ms. Womack, Generalized: One Model for Any Stocks

In M4.2 the model had **five named variables**, `m.Computer` to `m.Electronic`, and the stock names
were typed into `calc_risk`. That works for Ms. Womack, and nowhere else.

Tonight you write **one model that takes any CSV of monthly returns**: 5 stocks or 50, any tickers.
It is the same move you made in Studio 3 when `+=` loops replaced hard-coded terms.

1. **Part 1** reads the data into dictionaries (done for you).
2. **Part 2** is the model. You fill in two blocks: the **objective** and the **risk**.
3. **The test:** your model must reproduce Ms. Womack's answers from M4.2. When it passes, you trust it.
4. **Part 3** points the same model at 10 real stocks (five years of monthly returns), with no new code.

You may use Claude or another AI tool. A good prompt: paste the M4.2 `calc_risk` function and ask
it to rewrite the double loop using the `stocks` list and the `cov` dictionary from Part 1. Then make
sure you can explain every line, because the test only tells you *whether* it works.

## Import modules (download Pyomo and the ipopt solver)

Run this first, every time. It installs Pyomo and **ipopt**, the nonlinear solver we use in Module 4.
It takes about a minute.

In [ ]:
# install Pyomo and the ipopt solver in Colab (idaes-pse brings both)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

# this makes plots using pylab (matplotlib)
%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd

# import ALL NAMESPACES from pyomo
from pyomo.environ import *

# where the setup cell put the solver
ipopt_path = '/content/bin/ipopt'

# Part 1 · The data

One CSV, one row per month, one column per stock. The first column is the month.

In [ ]:
# ---------- the data file (Part 3: change this one line) ----------
data_folder = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio4/data/'
data_file = 'womack_returns.csv'
# data_file = 'stocks_monthly_returns.csv'     # Part 3: 10 real stocks, Oct 2021 - Sep 2026

returns_df = pd.read_csv(data_folder + data_file, index_col=0)
print(len(returns_df), "months,", len(returns_df.columns), "stocks")
returns_df.head()

**Soft-code it.** Three things come out of the data, and none of them mention a stock by name:

- `stocks` - the list of column names.
- `mean_return[stock]` - the average monthly return (the objective coefficients).
- `cov[stock_i][stock_j]` - the covariance matrix, as a dictionary of dictionaries (like `hours[dept][product]` in Studio 3.1).

In [ ]:
# the stocks are whatever columns the file has
stocks = list(returns_df.columns)

# average monthly return of each stock
mean_return = {}
for s in stocks:
    mean_return[s] = returns_df[s].mean()

# covariance matrix as a dictionary of dictionaries: cov[stock_i][stock_j]
cov_df = returns_df.cov()
cov = {}
for i in stocks:
    cov[i] = {}
    for j in stocks:
        cov[i][j] = cov_df.loc[i, j]

print("stocks:", stocks)
print()
print("average monthly return:")
for s in stocks:
    print("  ", s, ":", round(mean_return[s], 4))

In [ ]:
# the covariance matrix (the diagonal is each stock's variance)
cov_df.round(5)

**A note on "risk".** In this model, risk is the **variance** of the portfolio's monthly return,
the same number `calc_risk` computed in M4.2. Variance is in squared units, which is hard to picture.
Its square root, the **monthly volatility**, is easier: a variance of 0.0049 is a volatility of
0.07, so a typical month is about 7% above or below average. We report both.

# Part 2 · The model

`solve_portfolio(risk_limit)` builds and solves **one** model: the best average return you can get
without the portfolio's variance going over `risk_limit`. Part 2b calls it once per risk level to draw
the efficient frontier, exactly like M4.2.

**Fill in the two blocks marked TO DO.**

- **The objective** is the average return of the portfolio: each stock's `mean_return` times the share
  you hold, added up over all the stocks. One `+=` loop, like the objective in Studio 3.
- **The risk** is `calc_risk` from M4.2 with the names taken out: every pair of stocks `(i, j)`
  contributes `x[i] * cov[i][j] * x[j]`. Two loops, one inside the other.

In [ ]:
def solve_portfolio(risk_limit):
    model = ConcreteModel()

    # 1. decision variables: the share of the money in each stock, between 0 and 1
    model.x = Var(stocks, domain=NonNegativeReals, bounds=(0, 1))
    for s in stocks:
        model.x[s] = 1 / len(stocks)          # start from an even split (initialize!)

    # 2. objective: the average monthly return of the portfolio
    obj_expr = 0
    # ---------- TO DO: one loop over the stocks, adding mean_return times the share held ----------

    if type(obj_expr) == int:
        print("STOP: fill in the objective TO DO block first")
        return None
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)

    # 3. constraints
    model.constraints = ConstraintList()

    # 3a. invest all of the money
    budget = 0
    for s in stocks:
        budget += model.x[s]
    model.constraints.add(budget == 1)

    # 3b. risk: the covariance matrix weighted by what you hold, summed up (calc_risk from M4.2)
    risk_expr = 0
    # ---------- TO DO: two loops (i over stocks, j over stocks), adding x[i] * cov[i][j] * x[j] ----------

    if type(risk_expr) == int:
        print("STOP: fill in the risk TO DO block first")
        return None
    model.constraints.add(risk_expr <= risk_limit)

    # solve it
    results = SolverFactory('ipopt', executable=ipopt_path).solve(model)
    if results.solver.termination_condition != TerminationCondition.optimal:
        return None      # no portfolio is that safe

    weights = {}
    for s in stocks:
        weights[s] = model.x[s]()
    return {'return': model.portfolio_return(), 'risk': value(risk_expr), 'weights': weights}


print("solve_portfolio is ready (this cell only defines it - the next cells call it)")

## The test: does it reproduce Ms. Womack?

In M4.2, with the five Womack stocks, the best monthly return at three risk levels was:

| risk limit (variance) | best return in M4.2 |
|---|---|
| 0.002 | 0.0175 (about 1.75% a month) |
| 0.004 | 0.0208 |
| 0.006 | 0.0224 |

Run the cell. It only runs the test when `data_file` is the Womack file. **PASS on all three** means
your generalized model is the M4.2 model. If you see FAIL or an error, check your two TO DO blocks.

In [ ]:
# ---------- the test: your model vs. Ms. Womack from M4.2 ----------
expected = {0.002: 0.017542, 0.004: 0.020786, 0.006: 0.022374}

if data_file == 'womack_returns.csv':
    passed = 0
    for risk_limit in expected:
        answer = solve_portfolio(risk_limit)
        if answer is None:
            print("risk limit", risk_limit, ": no solution  <-- FAIL")
        elif abs(answer['return'] - expected[risk_limit]) < 0.00005:
            print("risk limit", risk_limit, ": return = %.4f  PASS" % answer['return'])
            passed += 1
        else:
            print("risk limit", risk_limit, ": return = %.4f, expected %.4f  <-- FAIL" % (answer['return'], expected[risk_limit]))
    print()
    print(passed, "of", len(expected), "passed")
else:
    print("The test only applies to the Womack data. You are on", data_file, "- carry on to the frontier.")

## 2b · The efficient frontier

Now solve the model at many risk levels, from very safe to "everything in the best stock". The range is
set **from the data**, so it works for any file:

- at the top, the variance of the stock with the highest average return (past that, more risk buys nothing);
- at the bottom, a small fraction of the smallest single-stock variance (the lowest levels may be impossible,
  and those are skipped).

In [ ]:
# ---------- the risk levels, set from the data ----------
best_stock = stocks[0]
for s in stocks:
    if mean_return[s] > mean_return[best_stock]:
        best_stock = s

smallest_variance = cov[stocks[0]][stocks[0]]
for s in stocks:
    if cov[s][s] < smallest_variance:
        smallest_variance = cov[s][s]

risk_levels = np.linspace(smallest_variance / 10, cov[best_stock][best_stock], 40)
print("highest average return:", best_stock)
print("risk levels from %.5f to %.5f" % (risk_levels[0], risk_levels[-1]))

In [ ]:
# ---------- solve once per risk level ----------
frontier = []
skipped = 0

for risk_limit in risk_levels:
    answer = solve_portfolio(risk_limit)
    if answer is None:
        skipped += 1
        continue
    row = {'risk limit': risk_limit, 'risk': answer['risk'], 'volatility': np.sqrt(answer['risk']), 'return': answer['return']}
    for s in stocks:
        row[s] = answer['weights'][s]
    frontier.append(row)

frontier_df = pd.DataFrame(frontier)
print(len(frontier_df), "portfolios on the frontier,", skipped, "risk levels too low to reach")
frontier_df.round(4)

In [ ]:
# ---------- the efficient frontier ----------
if len(frontier_df) == 0:
    print("No portfolios yet: finish the two TO DO blocks in Part 2 first.")
else:
    plot(frontier_df['risk'], frontier_df['return'], marker='o')

    # the single stocks, for comparison
    for s in stocks:
        scatter(cov[s][s], mean_return[s], color='gray')
        annotate(s, (cov[s][s], mean_return[s]), fontsize=8)

    # the S&P 500, for comparison (real data only)
    if data_file == 'stocks_monthly_returns.csv':
        spy = pd.read_csv(data_folder + 'spy_monthly_returns.csv', index_col=0)['SPY']
        scatter(spy.var(), spy.mean(), color='red', marker='*', s=200, label='S&P 500 (SPY)')
        legend()

    xlabel('Risk (variance of the monthly return)')
    ylabel('Average monthly return')
    title('The efficient frontier')
    show()

In [ ]:
# ---------- what to hold at each risk level ----------
if len(frontier_df) == 0:
    print("No portfolios yet: finish the two TO DO blocks in Part 2 first.")
else:
    stackplot(frontier_df['risk'], frontier_df[stocks].T.values, labels=stocks)
    xlabel('Risk (variance of the monthly return)')
    ylabel('Share of the money')
    title('The allocation along the frontier')
    legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)
    show()

# Part 3 · Real stocks, no new code

Go back to Part 1, put a `#` in front of `data_file = 'womack_returns.csv'`, remove the `#` from the
`stocks_monthly_returns.csv` line, and choose **Runtime → Run all**.

The file has five years of monthly returns (October 2021 to September 2026) for ten large companies,
one from each of ten sectors: Apple, JPMorgan Chase, Johnson & Johnson, ExxonMobil, Coca-Cola, Home
Depot, Caterpillar, NextEra Energy, Alphabet and Linde. The red star is the S&P 500 over the same months.

### Your turn

1. Did your model pass the Ms. Womack test? Which TO DO block was harder, and why?
2. On the real data, which stock does the frontier end on (all the money in one stock)? Is that a stock you
   would put *all* of anyone's savings in?
3. Find the portfolio at the safest end of the frontier. How many stocks does it hold, and what is its
   monthly volatility?
4. Is the S&P 500 star above, on, or below the frontier? What does that mean, and why shouldn't you trust it
   too much? (Hint: these are averages of the past five years.)
5. Pick one point on the frontier for Ms. Womack and explain your choice in two sentences.

## Appendix · How the data file was made

`stocks_monthly_returns.csv` came from Yahoo Finance with the `yfinance` package. Here is the code, if
you want to download your own tickers. Downloads from Colab sometimes get rate-limited, which is why
tonight's file is already saved for you.

In [ ]:
# OPTIONAL: download your own monthly returns (remove the # marks to run)
# !pip install -q yfinance
# import yfinance as yf
#
# my_tickers = ['AAPL', 'JPM', 'JNJ', 'XOM', 'KO', 'HD', 'CAT', 'NEE', 'GOOGL', 'LIN']
# prices = yf.download(my_tickers, start='2021-09-01', end='2026-10-01', interval='1mo', auto_adjust=True)['Close']
# my_returns = prices.pct_change().dropna()        # monthly return = percent change of the month-end price
# my_returns.index = my_returns.index.strftime('%Y-%m')
# my_returns.to_csv('my_returns.csv')
# my_returns.head()

## Bottom line

The model never mentions a stock by name. The data decides how many variables there are, the loops
build the objective and the risk, and a known answer (Ms. Womack) tells you the code is right before you
trust it on new data. That is the pattern for your final project.